## Project setup
You can replace it with your GitHub repository to make making changes easier.

In [10]:
!rm -rf cv-hw1-bilyi
!git clone https://github.com/bilyi1andrii/cv-hw1-bilyi.git
!cp -r cv-hw1-bilyi/* .

Cloning into 'cv-hw1-bilyi'...
remote: Enumerating objects: 241, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 241 (delta 135), reused 121 (delta 112), pack-reused 89 (from 1)
Receiving objects: 100% (241/241), 12.75 MiB | 11.20 MiB/s, done.
Resolving deltas: 100% (138/138), done.


### Data setup

In [ ]:
!rm -r iam_data
!mkdir -p iam_data
!mkdir -p iam_data/words
!mkdir -p iam_data/ascii
!gdown --id 16LUHxvrXzpHZGu2IuvHlfSneR3OdPsOY
!gdown --id 187Ww-EIPU6SNwMlKWB3_WDlr8lix3xJs
!tar -xzf words.tgz -C iam_data/words
!tar -xzf ascii.tgz -C iam_data/ascii

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=16LUHxvrXzpHZGu2IuvHlfSneR3OdPsOY
From (redirected): https://drive.google.com/uc?id=16LUHxvrXzpHZGu2IuvHlfSneR3OdPsOY&confirm=t&uuid=dfa469d4-777c-4a7a-ab4f-7ab3faf61de4
To: /content/words.tgz
100% 821M/821M [00:11<00:00, 74.6MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=187Ww-EIPU6SNwMlKWB3_WDlr8lix3xJs
To: /content/ascii.tgz
100% 2.61M/2.61M [00:00<00:00, 15.5MB/s]


In [ ]:
!ls ./iam_data/ascii/forms.txt

./iam_data/ascii/forms.txt


### Code update

In [ ]:
!sed -i 's|/path/to/iam_data/|./iam_data/|g' style_encoder_train.py

In [ ]:
!python style_encoder_train.py --epochs 5

## Fix Training Pipeline (5 points)

1. Fix the training pipeline.

2. Split the dataset into training and validation sets in advance using a fixed split.

3. Provide a summary describing:
  - what changes were made to the pipeline to make it work, and
  - how well the pipeline performs on the training and validation datasets.

4. Use the dataset https://www.kaggle.com/datasets/annyhnatiuk/ukrainian-handwritten-text
 to train a model for Ukrainian handwritten text recognition.

5. Submit link to Google Collab.

### Why it works now?
TODO What fixed

### Ukrainian Dataset

In [ ]:
import os
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split

In [2]:
# Downloading Dataset
kaggle_key = userdata.get('KAGGLE_API_TOKEN')

os.environ['KAGGLE_USERNAME'] = "andriibilyi"
os.environ['KAGGLE_KEY'] = kaggle_key

!kaggle datasets download -d annyhnatiuk/ukrainian-handwritten-text

Dataset URL: https://www.kaggle.com/datasets/annyhnatiuk/ukrainian-handwritten-text
License(s): CC-BY-SA-4.0
100% 3.68G/3.68G [00:33<00:00, 117MB/s]



In [3]:
!unzip -q ukrainian-handwritten-text.zip -d ukrainian_data

In [5]:
!head -n 5 ukrainian_data/METAFILE.tsv

filename	transcription
a01-001-0023-01.png	Що ж спонукає людей (незалежно
a01-001-0023-02.png	від того, є вони чи релігійними
a01-001-0023-03.png	паломниками, чи екскурсійно-
a01-001-0023-04.png	пізнавальними релігійними


In [11]:
# Splitting into train and val
df = pd.read_csv('ukrainian_data/METAFILE.tsv', sep='\t')

df['author_id'] = df['filename'].apply(lambda x: str(x).split('-')[2])

train_list = []
val_list = []

# Ensure the same number of writers on both sets
for author, group in df.groupby('author_id'):
    if len(group) > 1:
        t, v = train_test_split(group, test_size=0.2, random_state=42)
        train_list.append(t)
        val_list.append(v)
    else:
        train_list.append(group)


train_df = pd.concat(train_list).drop(columns=['author_id'])
val_df = pd.concat(val_list).drop(columns=['author_id'])

train_df.to_csv('ukrainian_data/train_split.csv', index=False)
val_df.to_csv('ukrainian_data/val_split.csv', index=False)

train_writers = len(set([str(f).split('-')[2] for f in train_df['filename']]))
val_writers = len(set([str(f).split('-')[2] for f in val_df['filename']]))

print(f"Train images: {len(train_df)} | Unique Writers: {train_writers}")
print(f"Val images: {len(val_df)} | Unique Writers: {val_writers}")

Train images: 29557 | Unique Writers: 331
Val images: 7554 | Unique Writers: 331


**Performance on Train/Val**

TODO

In [ ]:
!python style_encoder_train.py --dataset ukrainian --epochs 5

num_tokens 1
Number of character classes: 102
Loading fixed training split...
save_file ./IAM_dataset_PIL_style/train_line_Ukrainian.pt
Loading train images: [0/29557]
Loading train images: [2000/29557]
Loading train images: [4000/29557]
Loading train images: [6000/29557]
Loading train images: [8000/29557]
Loading train images: [10000/29557]
Loading train images: [12000/29557]
Loading train images: [14000/29557]
Loading train images: [16000/29557]
Loading train images: [18000/29557]
Loading train images: [20000/29557]
Loading train images: [22000/29557]
Loading train images: [24000/29557]
Loading train images: [26000/29557]
Loading train images: [28000/29557]
Number of writers 331
Max transcription length: 579
Loading fixed validation split...
save_file ./IAM_dataset_PIL_style/val_line_Ukrainian.pt
Loading val images: [0/7554]
Loading val images: [2000/7554]
Loading val images: [4000/7554]
Loading val images: [6000/7554]
Number of writers 331
Max transcription length: 462
Detected 331 